# Predictable Trends, Unpredictable Prices
## Full experimental pipeline — Google Colab runner

Runs every experiment end to end and packages the results for paper writing.

> **Before you start:** *Runtime → Change runtime type → GPU* (T4 suffices).
> Thirteen models are trained; on CPU that is most of a day, on GPU a couple
> of hours.

### Steps

| § | Step | Output |
|---|---|---|
| 1 | Environment, GPU, clone Time-Series-Library | — |
| 2 | **Data collection** (Yahoo / yfinance): forecast + portfolio universes | `results/design/` |
| 3 | **Splits and folds** — walk-forward design figure | `results/design/` |
| 4 | **Kalman filter + leakage audit** with plots | `results/leakage/` |
| 5 | **Training** — 13 models incl. LSTM, DLinear, TimeFilter | `results/forecast/` |
| 6 | **Deep result analysis** + 12 prediction windows of the best model | `results/forecast/` |
| 7 | **Strategy** on the best 4 models (MCAP, blended signal) | `results/strategy*/` |
| 8 | **Per-year return attribution** — why each year was good or bad | `results/year_analysis/` |
| 9 | **FLIP ablation** — reverse the model's call | `results/strategy/` |
| 10 | **Export** — every table, figure, prediction and `STEPS.md` | `results/paper_export/` |

Everything lands in `results/`, and §10 produces a single archive to download.

**The long step (§5) is resumable** — if Colab disconnects, re-run the cell and
it continues from the last finished model instead of starting over.

---
## 1. Environment

In [ ]:
import subprocess, sys, os, time, shutil, zipfile, glob
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "xgboost>=2.0", "lightgbm>=4.0", "statsmodels>=0.14",
                "einops>=0.7", "yfinance", "PyWavelets>=1.4"], check=True)

import torch
GPU = torch.cuda.is_available()
print("torch :", torch.__version__)
print("GPU   :", torch.cuda.get_device_name(0) if GPU else "NONE (CPU only)")
if GPU:
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM  : {vram:.1f} GB")
    if "T4" in name:
        print("\nT4 detected. Expected wall-clock on this runtime:")
        print("   data + design + leakage + filters   ~5 min")
        print("   forecast, 11 models x 3 seeds       ~2-3 h")
        print("   horizons sweep, SLOPE + LR          ~10 min")
        print("   horizons sweep with GRU (optional)  ~3-4 h")
        print("   strategy + folds + years + export   ~25 min")
        print("\nColab free tier disconnects after ~12 h and idles out after 90 min.")
        print("Every step caches to results/, so a reconnect resumes rather than restarts.")
        print("\nCrossformer and FEDformer are commented out in config.MODELS: at three")
        print("seeds FEDformer alone would outweigh the other ten models combined.")
        print("Uncomment them there if you want the full thirteen.")
else:
    print("\n!! No GPU. Runtime -> Change runtime type -> T4 GPU.")
    print("   The 13-model sweep takes many hours on CPU.")

### 1b. Get the project

Either upload `kalman-trend-pred.zip`, or set `GIT_URL` below to clone it.

In [ ]:
GIT_URL = ""          # e.g. "https://github.com/<user>/kalman-trend-pred.git"

WORK = Path("/content/work")
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)

if GIT_URL:
    subprocess.run(["git", "clone", "-q", GIT_URL, str(WORK / "repo")], check=True)
else:
    from google.colab import files
    up = files.upload()                       # pick kalman-trend-pred.zip
    with zipfile.ZipFile(next(iter(up))) as z:
        z.extractall(WORK)

cands = [p.parent for p in WORK.rglob("run_all.py")]
if not cands:
    raise SystemExit("run_all.py not found — is this the right archive?")
PROJ = cands[0]
os.chdir(PROJ)
sys.path.insert(0, str(PROJ))
print("project:", PROJ)
print(sorted(p.name for p in PROJ.glob("*.py")))

### 1c. Clone the Time-Series-Library

`DLinear`, `PatchTST`, `TimeMixer`, `TimeFilter`, `Crossformer` and `FEDformer`
are the **official** implementations, vendored unmodified at a pinned commit.

In [ ]:
TSLIB_COMMIT = "4e938a1767106324dd753b2a44832bf870a0252e"
tslib = PROJ / "third_party" / "Time-Series-Library"

if not (tslib / "models" / "PatchTST.py").exists():
    tslib.parent.mkdir(parents=True, exist_ok=True)
    print("cloning thuml/Time-Series-Library ...")
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/thuml/Time-Series-Library.git", str(tslib)],
                   check=True)
    subprocess.run(["git", "-C", str(tslib), "checkout", "-q", TSLIB_COMMIT], check=True)

head = subprocess.run(["git", "-C", str(tslib), "rev-parse", "HEAD"],
                      capture_output=True, text=True).stdout.strip()
print("HEAD  :", head)
print("pinned:", TSLIB_COMMIT, "| match:", head == TSLIB_COMMIT)
for m in ["DLinear", "PatchTST", "TimeMixer", "TimeFilter", "Crossformer", "FEDformer"]:
    print(f"  {m:12}", "ok" if (tslib / 'models' / f'{m}.py').exists() else "MISSING")

In [ ]:
import importlib, config, models
importlib.reload(config)

print("models        :", len(config.MODELS), config.MODELS)
print("device        :", models.DEVICE)
print("horizon/window: h =", config.H, ", L =", config.L)
print("folds         :", len(config.FOLDS), "->", config.FOLDS[0][0], "..", config.FOLDS[-1][1])
print("forecast univ :", list(config.ASSETS))
print("portfolio univ:", list(config.ORIGINAL_ASSETS))
print("costs         :", f"{config.COST*1e4:.0f} bps/side + {config.FUND:.0%} p.a. crypto short funding")

# --- optional: shorten the run for a plumbing check ---
# config.MODELS = ["LR", "RF", "XGB", "LGBM", "ARIMA", "DLinear"]
# config.MODELS = [m for m in config.MODELS if m not in config.SLOW_MODELS]

In [ ]:
def run_step(step, timeout=12 * 3600, **env_extra):
    """Run one pipeline step in a subprocess, streaming its output."""
    print(f"\n{'='*72}\n>>> {step}\n{'='*72}", flush=True)
    t0 = time.time()
    env = dict(os.environ, PYTHONUNBUFFERED="1", **env_extra)
    p = subprocess.Popen([sys.executable, "run_all.py", step], env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait(timeout=timeout)
    print(f"<<< {step}: exit {p.returncode} in {(time.time()-t0)/60:.1f} min", flush=True)
    return p.returncode


from IPython.display import Image, display, Markdown
def show_fig(rel, width=980):
    f = PROJ / "results" / rel
    display(Image(filename=str(f), width=width)) if f.exists() else print(f"[missing] {rel}")

import pandas as pd, numpy as np
pd.set_option("display.width", 200)
import config as _cfg
cfg_models = list(_cfg.MODELS)
print("helpers ready |", len(cfg_models), "models:", ", ".join(cfg_models))

---
## 2. Data collection

Daily closes for two **disjoint** universes — five assets (one per class) used
only to compare forecasters, and ten used only to trade. Keeping them disjoint
means no asset both selects a model and is traded by it.

The loader tries Yahoo's chart API with explicit epoch bounds (which preserves
true daily granularity — `range=max` silently downsamples long histories to
monthly), and falls back to `yfinance` if that IP is rate-limited.

In [ ]:
run_step("data")

from data import load_asset
rows = []
for n in list(config.ASSETS) + list(config.ORIGINAL_ASSETS):
    d = load_asset(n)
    rows.append(dict(asset=n, universe="forecast" if n in config.ASSETS else "portfolio",
                     n_obs=len(d), start=d.date.min().date(), end=d.date.max().date()))
inv = pd.DataFrame(rows)
(PROJ / "results").mkdir(exist_ok=True)
inv.to_csv(PROJ / "results" / "design" / "data_inventory.csv", index=False) if (PROJ/"results"/"design").exists() else None
inv

---
## 3. Splits and folds

Anchored walk-forward: five non-overlapping annual test folds. Training and
validation come only from data at or before the fold's test start, minus an
**8-day embargo** ($h+1$) so no training label can overlap the test window.

The figure is drawn from the spans the split code actually returns, so it
cannot drift from the implementation.

In [ ]:
run_step("design")
show_fig("design/experimental_design.png")
display(pd.read_csv(PROJ / "results" / "design" / "fold_spans.csv"))

---
## 4. Kalman filter and the leakage audit

The learning target is the $h$-step change of a **causal local-linear-trend
Kalman filter** (state = level + velocity). Because the whole study rests on
that target being causal, it is audited rather than asserted — on `LTC`, one of
the forecast-evaluation assets:

- **(a) Influence kernel** $\partial \ell_{t_0}/\partial p_{t_0+k}$ — must be
  exactly zero for every future bar $k>0$.
- **(b) Future randomisation** — replace every price after $t_0$ with noise and
  recompute; the filtered series must be *bit-identical* up to $t_0$.

The subtle failure this catches: the filter's volatility-adaptive measurement
noise is normalised by a constant computed from **pre-test data only**.
Normalising over the full sample is a real look-ahead leak that leaves the
filter itself looking perfectly causal.

In [ ]:
run_step("leakage")
show_fig("leakage/kalman_leakage.png")

import json
audit = json.load(open(PROJ / "results" / "leakage" / "audit.json"))
print(json.dumps(audit, indent=2))
print("\nBoth quantities must be exactly 0 — not merely small.")

In [ ]:
# The filter itself, on a training asset, so the target is visible
import matplotlib.pyplot as plt
from config import PLOT_RC
from targets import kalman_causal
plt.rcParams.update(PLOT_RC)

ASSET = "LTC"
df = load_asset(ASSET)
lp = df.logprice.values.astype(float)
ts = config.FOLDS[-1][0]
norm_end = int(np.searchsorted(df.date.values, np.datetime64(ts)))
level, vel = kalman_causal(lp, norm_end)
seg = slice(len(lp) - 700, len(lp))

fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(11, 6), sharex=True,
                               gridspec_kw={"height_ratios": [2, 1]})
ax0.plot(df.date.values[seg], lp[seg], color=".65", lw=0.9, label="log price $p_t$")
ax0.plot(df.date.values[seg], level[seg], color="crimson", lw=1.7,
         label=r"filtered level $\ell_t$")
ax0.set_ylabel("log price"); ax0.legend()
ax0.set_title(f"(a) {ASSET}: causal local-linear-trend Kalman filter and the observed price")
ax1.plot(df.date.values[seg], vel[seg], color="navy", lw=1.2)
ax1.axhline(0, color="k", lw=0.8)
ax1.set_ylabel(r"velocity $v_t$"); ax1.set_xlabel("date")
ax1.set_title(r"(b) Filtered velocity state, whose persistence makes the $h$-step change learnable")
fig.tight_layout()
out = PROJ / "results" / "design" / f"kalman_state_{ASSET}.png"
out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out); plt.show()

---

## 4b. The same audit, applied to nine filters

The audit above clears the one target this project uses. That is necessary but
it is not an argument, because a test that has only ever been run on a
transform known to pass tells you nothing about whether it can catch anything.

So we run it over the whole registry in `filters.py`: four transforms that
should be causal (Kalman local linear trend, Kalman local level, EMA, and
Savitzky-Golay read at the trailing edge) and five that should not (centred
moving average, centred Savitzky-Golay, Hodrick-Prescott, wavelet denoising,
and a zero-phase Butterworth). The last five are not strawmen. They are the
smoothers this literature actually applies, and each one is applied the way it
is usually applied, to the whole series before splitting.

A filter passes only if **both** tests return exactly `0.0`. No tolerance.

In [ ]:
run_step("filters")
show_fig("leakage/filter_audit.png", width=1100)

FA = pd.read_csv(PROJ / "results" / "leakage" / "filter_audit.csv")
cols = ["label", "expected_causal", "max_future_influence",
        "max_pre_cut_diff", "PASSED"]
display(FA[cols].style.format({
    "max_future_influence": "{:.3e}", "max_pre_cut_diff": "{:.3e}"}).hide(axis="index"))

n_ok = int(FA.PASSED.sum())
print(f"\n{n_ok} of {len(FA)} filters are causal; {len(FA)-n_ok} read the future.")
print(f"every filter matched its expected label: {bool(FA.matches_expectation.all())}")
print("\nThe causal four return exactly 0.0, not a small number. The other five")
print("are caught by both tests independently, which is the point of running two.")

---
## 5. Model training

Thirteen models across five families, all under an **identical protocol** —
same inputs, same target, same loss, same purged splits, same early stopping —
so "which model wins" is not a comparison inside one inductive bias.

| family | models |
|---|---|
| tabular linear | `LR` |
| tabular trees | `RF`, `XGB`, `LGBM` |
| classical state space | `ARIMA` |
| recurrent | `GRU`, `LSTM` |
| linear / transformer sequence (official TSLib) | `DLinear`, `PatchTST`, `TimeMixer`, `TimeFilter`, `Crossformer`, `FEDformer` |

**Resumable:** `FORECAST_RESUME=1` skips any model already on disk.

In [ ]:
# FORECAST_WORKERS fits the 25 (asset, fold) cells of each model at once.
# One fit uses ~0.6 GB of the T4's 15 GB and is launch-latency bound, not
# compute bound, so concurrency is where the speed is. Verified to reproduce
# the serial predictions exactly: it changes the wall clock and nothing else.
#   0 = auto-size from GPU memory   1 = serial (previous behaviour)
FORECAST_WORKERS = 0

rc = run_step("forecast", FORECAST_RESUME="1",
              FORECAST_WORKERS=str(FORECAST_WORKERS))

---
## 6. Deep analysis of the forecast results

Three views: the model ranking, the **central control** (the same prediction
scored against the filtered target *and* against real price), and twelve
prediction windows from the best model.

In [ ]:
FC = PROJ / "results" / "forecast"
comp = pd.read_csv(FC / "model_comparison.csv", index_col=0)
best = json.load(open(FC / "best_model.json"))["best_model"]
print("Mean direction accuracy vs the causal Kalman trend\n")
display(comp.round(3))
print(f"\nBEST MODEL: {best}")
show_fig("forecast/model_comparison.png")

In [ ]:
# THE CENTRAL CONTROL: one prediction vector, three different truths.
rows = []
for f in sorted(FC.glob("*_predictions.csv")):
    d = pd.read_csv(f)
    s = np.sign(d.y_pred)
    rows.append(dict(model=d.model.iloc[0], n=len(d),
                     DA_vs_kalman_trend=(s == np.sign(d.y_true)).mean(),
                     DA_vs_raw_h_day=(s == np.sign(d.y_true_price_h)).mean(),
                     DA_vs_next_day=(s == np.sign(d.y_true_price_1)).mean()))
T3 = pd.DataFrame(rows).sort_values("DA_vs_kalman_trend", ascending=False)
T3.to_csv(FC / "three_truths.csv", index=False)
display(T3.round(4))
print("Accuracy against the SMOOTHED target is high; against the return a position")
print("actually earns on it sits at the coin flip. The gap is the finding.")

fig, ax = plt.subplots(figsize=(9, 4.4))
x = np.arange(len(T3)); w = 0.27
ax.bar(x - w, T3.DA_vs_kalman_trend, w, label="vs causal Kalman trend", color="#4C72B0")
ax.bar(x, T3.DA_vs_raw_h_day, w, label="vs raw $h$-day price change", color="#DD8452")
ax.bar(x + w, T3.DA_vs_next_day, w, label="vs next-day return", color="#C44E52")
ax.axhline(0.5, color="k", ls="--", lw=1.1, label="coin flip (50%)")
ax.set_xticks(x, T3.model, rotation=25, ha="right")
ax.set_xlabel("model"); ax.set_ylabel("direction accuracy")
ax.set_title("The same forecast scored against three targets:\n"
             "high accuracy on the filtered trend does not transfer to price")
ax.legend(ncol=2)
fig.tight_layout(); fig.savefig(FC / "three_truths.png"); plt.show()

In [ ]:
# Twelve prediction windows from the best model, evenly spaced across the last fold
import forecast as fmod
importlib.reload(fmod)
fmod.plot_windows(best, asset="LTC", n=12, nrow=3, ncol=4)
show_fig(f"forecast/pred12_LTC_{best}.png")
display(pd.read_csv(FC / f"windows_LTC_{best}.csv").round(4))

---

## 6b. Does the result generalise? Nine filters x three horizons

Everything so far is measured with one smoother at one horizon. If the effect
is really a property of smoothing rather than of the Kalman filter at h = 7,
it has to survive changing both.

This sweep re-runs the three-truths comparison over all nine filters and over
h in {3, 7, 14}. Two forecasters are scored on every cell:

- **SLOPE**, `h * (level_t - level_{t-1})`, which fits nothing. It is the
  sharpest statement of the argument: if a rule with no parameters tracks the
  fitted models, the accuracy belongs to the target.
- **LR**, the flattened-window linear model, within 0.001 of the best model in
  the main table and almost free to fit.

Set `GRU_SWEEP = True` to add the best model as well. On a T4 that turns a
ten-minute cell into three or four hours, so it is off by default.

Watch two columns. For the causal filters, price accuracy should sit at chance
no matter what h is. For the leaky ones, it should not — and that is the
mechanism behind published accuracies that look too good.

In [ ]:
GRU_SWEEP = False     # True adds the GRU to every cell: ~3-4 h on a T4

rc = run_step("horizons", HORIZONS_GRU="1" if GRU_SWEEP else "0")
show_fig("horizons/filter_horizon_sweep.png", width=1200)

In [ ]:
HZ = pd.read_csv(PROJ / "results" / "horizons" / "filter_horizon_sweep.csv")

piv = HZ[HZ.model == "LR"].pivot_table(
    index=["causal", "filter"], columns="h",
    values=["DA_trend", "DA_price_1"]).round(3)
print("LR: accuracy on the filtered target, and on the next-day return\n")
display(piv)

print("\nThe question that decides whether the effect generalises:")
for causal in (True, False):
    s = HZ[HZ.causal == causal]
    n = int((s.DA_price_1 > 0.5).sum())
    tag = "causal" if causal else "leaky "
    print(f"  {tag} filters: {n:2}/{len(s)} cells beat 0.5 against the next-day "
          f"return  (max {s.DA_price_1.max():.4f})")

In [ ]:
# The free baseline against the fitted model, on the target itself.
p = HZ.pivot_table(index=["causal", "filter", "h"], columns="model",
                   values="DA_trend")
if "LR" in p.columns:
    p["SLOPE - LR"] = p["SLOPE"] - p["LR"]
    display(p.round(4))
    d = p["SLOPE - LR"]
    print(f"SLOPE beats the fitted model in {int((d > 0).sum())} of {len(d)} cells; "
          f"mean difference {d.mean():+.4f}.")
    print("On the causal filters a rule with no parameters is competitive with a")
    print("fitted one. On the leaky filters the fitted model pulls ahead, because")
    print("there the leak is real structure and a model can learn it.")

---

## 6c. The generalised study: every model, every causal transform, every horizon

Section 6b widened the grid but only with SLOPE and LR. That leaves one
objection open, and it is a fair one: perhaps the larger models would have
behaved differently, and the effect is an artefact of using weak forecasters.

This section closes it by running the **whole model set** over every transform
that passed the causality audit and every horizon:

    4 causal filters  x  3 horizons  x  11 models  x  5 assets  x  5 folds

The leaky filters are excluded on purpose. Section 4b already established what
they do; spending GPU hours fitting eleven models against a target that reads
the future would only produce impressive numbers we would then have to explain
away.

### Why this needs its own runner

You will have noticed the GPU sitting at roughly **0.6 GB of 15 GB** during
training. That is not spare capacity waiting for a bigger model. It is what this
workload looks like: the nets are small (`d_model=64`, a 30-step window) and the
batch is 64 rows, so every kernel is tiny and the card spends most of its time
between launches rather than inside them. Raising the batch size would improve
occupancy and simultaneously change the optimisation, and therefore every number
in the study, so it is not on the table.

The fix that costs nothing in fidelity is **concurrency**. Each
(filter, horizon, model, asset, fold) cell is completely independent, so we run
many at once, each in its own process with its own CUDA context, and let the
card interleave them. Every cell writes its own small CSV, so the sweep is
resumable at cell granularity and workers never contend for a file.

In [ ]:
# What does one fit actually cost on this card, and how many fit alongside it?
import subprocess, sys
print(subprocess.run([sys.executable, "gpu_probe.py"], cwd=str(PROJ),
                     capture_output=True, text=True).stdout)

Pick the worker count from the probe above. A T4 usually lands on 8 to 12.

Two things to expect. Scaling is sub-linear: the first few workers help a lot
because they hide launch latency, later ones less as the SMs fill. And the
`--filters causal --horizons 3,7,14` default is 3300 cells; start with a smaller
slice if you want to see the shape of the result before committing the hours.

In [ ]:
WORKERS   = 0           # 0 = auto-size from GPU memory (see the probe above)
FILTERS   = "causal"    # "causal" (4), "all" (9), or e.g. "kalman,ema"
HORIZONS  = "3,7,14"

# Start with a slice to confirm the plumbing, then set QUICK = False for the
# full grid. QUICK is 3 models instead of 11, so roughly a seventh of the work.
QUICK = True
MODELS_ARG = "LR,DLinear,GRU" if QUICK else ",".join(cfg_models)

n_cells = (len(FILTERS.split(",")) if FILTERS not in ("causal", "all")
           else (4 if FILTERS == "causal" else 9)) \
          * len(HORIZONS.split(",")) * len(MODELS_ARG.split(",")) * 5 * 5
print(f"{n_cells} cells to run (already-finished cells are skipped)\n")

rc = run_step("sweep", SWEEP_FILTERS=FILTERS, SWEEP_HORIZONS=HORIZONS,
              SWEEP_MODELS=MODELS_ARG, SWEEP_WORKERS=str(WORKERS))

In [ ]:
import pandas as pd
SW = pd.read_csv(PROJ / "results" / "sweep" / "sweep_three_truths.csv")

print("Accuracy on the filtered target vs on the next-day return\n")
display(SW.pivot_table(index=["filter", "model"], columns="h",
                       values=["DA_trend", "DA_price_1"]).round(3))

print("\nThe generalisation claim, in one line per horizon:")
for h in sorted(SW.h.unique()):
    s = SW[SW.h == h]
    print(f"  h={h:<3} trend {s.DA_trend.mean():.3f}  "
          f"price {s.DA_price_1.mean():.3f}  "
          f"cells beating 0.5 on price: {int((s.DA_price_1>0.5).sum())}/{len(s)}")

In [ ]:
# The point the whole paper turns on, now across the full grid:
# does a rule with zero parameters keep up with the fitted models?
piv = SW.pivot_table(index=["filter", "h"], columns="model", values="DA_trend")
fitted = [c for c in piv.columns if c != "SLOPE"]
if "SLOPE" in piv.columns and fitted:
    piv["best fitted"] = piv[fitted].max(axis=1)
    piv["SLOPE - best"] = piv["SLOPE"] - piv["best fitted"]
    display(piv[["SLOPE", "best fitted", "SLOPE - best"]].round(4))
    d = piv["SLOPE - best"]
    print(f"SLOPE matches or beats the best fitted model in "
          f"{int((d >= 0).sum())} of {len(d)} (filter, horizon) combinations.")

---
## 7. Strategy

Equal-weight book over the 10 portfolio assets. Execution stack, in order:
SMA-200 regime gate → turn-classifier exit (threshold chosen on **validation**)
→ causal per-asset volatility target → 60/40 blend of prediction and regime
signal → book-level volatility target. Costs: **10 bps per side** on every
position change, plus **10% p.a.** funding on short crypto.

In [ ]:
run_step("strategy")
ST = PROJ / "results" / "strategy"
print("\nPooled results by signal arm\n")
display(pd.read_csv(ST / "ablation.csv").round(3))
print("\nPer-fold Sharpe\n")
display(pd.read_csv(ST / "fold_sharpe.csv", index_col=0).round(3))
show_fig("strategy/equity_mcap.png")

In [ ]:
run_step("folds")
show_fig("fold_analysis/fold_signal_sharpe.png")
show_fig("fold_analysis/fold_gross_vs_net.png")

### 7b. Strategy on the best four models

Takes the **four best forecasters by direction accuracy** — read from the
forecast results, not hardcoded — and runs each through the *identical*
execution stack, reporting the blended (`BLEND`) and unblended (`PRED`) arms
plus the `FLIP` ablation and a permutation null per model.

This answers a question the accuracy table cannot: **is direction accuracy a
useful criterion for choosing a trading signal?**

> Expensive — it refits each model over the 10-asset portfolio universe.
> Fits are cached, so re-running to add an arm is free.

In [ ]:
RUN_MODEL_STRATEGY = True     # set False to skip

if RUN_MODEL_STRATEGY:
    p = subprocess.Popen([sys.executable, "strategy_models.py"],
                         env=dict(os.environ, PYTHONUNBUFFERED="1"),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    show_fig("strategy_models/equity_pred_flip.png")
    show_fig("strategy_models/ablation_pred_flip.png")
else:
    print("skipped")

---
## 8. Per-year return attribution

Sharpe hides the two things that produce it. A weak fold can mean the strategy
earned nothing (**numerator**) or earned normally while carrying far more risk
(**denominator**) — and the remedies are opposite. Each fold is therefore split
into return, volatility, long and short leg contributions, cost drag, and the
*market context*: what the assets themselves did, how correlated they were, how
cleanly they trended, and how many were profitable.

Market-context columns are properties of the **data**, not the strategy, so a
bad year can be attributed to a hostile market — or not, which is the more
serious finding.

In [ ]:
run_step("years")
YA = PROJ / "results" / "year_analysis"
Y = pd.read_csv(YA / "year_attribution.csv", index_col=0)
print("Return and risk by fold\n")
display(Y[["return_pct", "ann_vol", "sharpe", "max_dd_pct"]].round(2))
print("\nWhere the return came from (%)\n")
display(Y[["long_leg_pct", "short_leg_pct", "cost_drag_pct", "turnover"]].round(2))
print("\nWhat the market was doing (properties of the data)\n")
display(Y[["asset_drift_pct", "asset_dispersion", "asset_ann_vol",
           "avg_pair_corr", "trend_efficiency", "breadth"]].round(3))
show_fig("year_analysis/year_attribution.png")

In [ ]:
# Generated per-fold narrative — why each year was good or bad
display(Markdown((YA / "year_read.md").read_text(encoding="utf-8")))

---
## 9. FLIP ablation

The decisive test of whether the model's *sign* carries information. The whole
execution stack is held fixed and **only the directional call is replaced**:

| arm | signal |
|---|---|
| `PRED` | $\mathrm{sign}(\hat y_t)$ — the model's call |
| `FLIP` | $-\mathrm{sign}(\hat y_t)$ — the exact inverse |
| `LONG` | $+1$ always |
| `REGIME` | the SMA-200 state, no ML |
| `RANDOM` | random $\pm 1$, 20 draws (permutation null) |

If the prediction has real directional content, reversing it must **hurt**.
Comparisons use a paired stationary bootstrap on identical days, so both arms
see the same resample indices.

In [ ]:
abl = pd.read_csv(ST / "ablation.csv")
pp = pd.read_csv(ST / "paired_proofs.csv", index_col=0)
display(abl.round(3))
print("\nPaired stationary bootstrap (800 resamples, mean block 20 days)\n")
display(pp.round(3))

if "PRED_vs_FLIP" in pp.index:
    r = pp.loc["PRED_vs_FLIP"]
    verdict = ("entirely above zero — the sign carries real information"
               if r.ci_lo > 0 else "straddles zero — not demonstrated")
    print(f"\nPRED vs FLIP: dSharpe {r.d_sharpe:+.3f}, 95% CI "
          f"[{r.ci_lo:+.3f}, {r.ci_hi:+.3f}] -> {verdict}")

show_fig("strategy/ablation_bars.png")
show_fig("strategy/ablation_equity.png")

In [ ]:
# Caveat worth stating in the paper: the regime gate clamps direction, so
# flipping the prediction converts LONG exposure into FLAT exposure in bull
# regimes rather than into short exposure. The PRED-FLIP gap therefore partly
# measures forgone beta, not purely signal skill.
run_step("voltarget")
show_fig("vol_target/parameter_surfaces.png")
display(pd.read_csv(PROJ / "results" / "vol_target" / "controls.csv").round(3))

---
## 10. Export everything

Consolidates every table, figure and row-level file into `results/paper_export/`
with a `MANIFEST.csv`, a `config_snapshot.json`, an `environment.json`, and
**`STEPS.md`** — a generated write-up of every step with its headline numbers
read from the files, ready to feed the paper.

In [ ]:
run_step("export")
EX = PROJ / "results" / "paper_export"
man = pd.read_csv(EX / "MANIFEST.csv")
print(f"{len(man)} artefacts\n")
display(man[["file", "rows", "description"]])

In [ ]:
display(Markdown((EX / "STEPS.md").read_text(encoding="utf-8")))

In [ ]:
print((PROJ / "results" / "ACCEPTANCE.md").read_text(encoding="utf-8"))

In [ ]:
for p in sorted(EX.glob("fig__*.png")):
    print(f"\n--- {p.name} ---")
    display(Image(filename=str(p), width=980))

### Download

`kalman-trend-results.zip` contains the consolidated export **and** the raw
`results/` tree — every table, figure, prediction (`y_true` vs `y_pred`) and
position. This is the file to send back for the paper-writing stage.

In [ ]:
from google.colab import files
zp = Path("/content/kalman-trend-results.zip")
if zp.exists():
    zp.unlink()
shutil.make_archive(str(zp.with_suffix("")), "zip", root_dir=PROJ, base_dir="results")
print(f"{zp.name}  ({zp.stat().st_size/1e6:.1f} MB)")
files.download(str(zp))